In [2]:
import sys
sys.path.insert(0, '/home/stavz/masters/gdc/APM')  # Add parent directory to path

import pandas as pd
from pipeline.config import PATHS, PRIMARY_GENES, UCSC_RNA_SEQ_GENE_SYMBOL_CHANGE
from pipeline.RNA_exp import normalize_expression_mat
import re


rna = pd.read_csv(PATHS.rna_expression_raw, sep="\t")
rna = normalize_expression_mat(rna)
immune_sub = pd.read_csv(r"\home\stavz\masters\gdc\APM\annotations\BRCA_immune_subtypes.tsv", sep="\t")
brca = pd.read_csv(r"\home\stavz\masters\gdc\APM\annotations\BRCA_clinical", sep="\t")
PAM50 = pd.read_csv(r"\home\stavz\masters\gdc\APM\annotations\BRCA_PAM50.tsv", sep="\t")
purity = pd.read_excel(
    r"/home/stavz/masters/gdc/APM/data/RNAexp_TCGA/Purity_2015.xlsx", header = 3
    
)

# rna.set_index("gene", inplace=True)
# immune_sub.set_index("TCGA Participant Barcode", inplace=True)
# mapping = {
#     sample: immune_sub.loc["-".join(sample.split("-")[:3]), "TCGA Subtype"]
#     for sample in rna.columns if "-".join(sample.split("-")[:3]) in immune_sub.index and sample.split("-")[-1] == "01"}


c:\Users\stavz\AppData\Local\Programs\Python\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [3]:
# creating primary tumor suffix for thprnsson ids
immune_sub["TCGA Participant Barcode"] = immune_sub["TCGA Participant Barcode"].apply(lambda x: x + "-01")

# mapping thornsson subtypes to pam50 table
PAM50["thornsson_subtype"] = PAM50["sample_id"].map(immune_sub.set_index("TCGA Participant Barcode")["TCGA Subtype"])
PAM50["thornsson_subtype"] = PAM50["thornsson_subtype"].str.split(".").str[1]

PAM50["PAM50_final"] = (
    PAM50["PAM50Call_RNAseq"]
      .combine_first(PAM50["thornsson_subtype"])
      .combine_first(PAM50["PAM50_recomputed"])
)

# setting RNAseq_call as first priority
PAM50["PAM50_final"] = PAM50["PAM50Call_RNAseq"]

# setting thornsson subtypes as second priority where Xena RNAseq call is missing
PAM50[PAM50["PAM50_final"].isna()]["PAM50_final"] = PAM50[PAM50["PAM50Call_RNAseq"].isna()]["thornsson_subtype"]

# setting recomputed as third priority
PAM50[PAM50["PAM50_final"].isna()]["PAM50_final"] = PAM50[PAM50["PAM50_final"].isna()]["PAM50_recomputed"]

# setting all normal as "Normal"
PAM50.loc[PAM50["sample_id"].str.split("-").str[-1] == "11","PAM50_final"] = "Normal"



# -------adding purity to PAM50-------

# key for PAM50 index: "TCGA-3C-AAAU-01"
PAM50_key = PAM50.index.to_series().str.extract(r'^(TCGA-[^-]+-[^-]+-\d\d)')[0]

# key + aliquot for brca "Sample ID": "TCGA-3C-AAAU-01A" -> key="...-01", aliquot="A"
brca_tmp = brca[["Sample ID", "CPE"]].copy()
brca_tmp["key"] = brca_tmp["Sample ID"].str.extract(r'^(TCGA-[^-]+-[^-]+-\d\d)')[0]
brca_tmp["aliquot"] = brca_tmp["Sample ID"].str.extract(r'^TCGA-[^-]+-[^-]+-\d\d([A-Z])')[0]

# priority order
order = {"A": 0, "B": 1, "C": 2}
brca_tmp["aliquot_rank"] = brca_tmp["aliquot"].map(order).fillna(99)

# pick best-ranked aliquot per key
brca_best = (
    brca_tmp.sort_values(["key", "aliquot_rank"])
            .drop_duplicates("key", keep="first")
            .set_index("key")["CPE"]
)

# map into PAM50
PAM50["CPE"] = PAM50_key.map(brca_best).to_numpy()




# -------saving short PAM50 table-------
# PAM50.drop(columns=["sample_short", "sampleID"], inplace=True)
# PAM50.to_csv(PATHS.immune_subtype_annotations, sep="\t", index=False)

# #saving thornsson full subtype table
# immune_sub["PAM50_final"].map(PAM50.set_index("sample_id")["PAM50_final"])
# immune_sub.to_csv(r"\home\stavz\masters\gdc\APM\annotations\Thornsson_immune_table.tsv", sep="\t", index=False)


C:\Users\stavz\AppData\Local\Temp\ipykernel_37452\1300176769.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  PAM50[PAM50["PAM50_final"].isna()]["PAM50_final"] = PAM50[PAM50["PAM50Call_RNAseq"].isna()]["thornsson_subtype"]
C:\Users\stavz\AppData\Local\Temp\ipykernel_37452\1300176769.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  PAM50[PAM50["PAM50_final"].isna()]["PAM50_final"] = PAM50[PAM50["PAM50_final"].isna()]["PAM50_recomputed"]


AttributeError: Can only use .str accessor with string values!

In [5]:
advanced = pd.read_csv(PATHS.immune_subtype_annotations, sep="\t")

In [9]:
advanced["sample_id"] = PAM50["sample_id"]

In [12]:
ordered_cols = ["sample_id"] + [col for col in advanced.columns if col != "sample_id"]
advanced = advanced[ordered_cols]

In [14]:
advanced.to_csv(PATHS.immune_subtype_annotations, sep="\t", index=False)